In [8]:
from pathlib import Path

# Base folder
base_folder = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded")

# List to store found .tif paths
tif_files = []

# Prefixes
scenarios = ["cc45", "cc85"]

# Allowed suffixes
allowed_suffixes = ["y", "sb2y", "sm2y", "sb2rb1y", "sm2rb1y", "sb2rb3y"]
probability = "p50"

# Build folder names and search inside each
for scenario in scenarios:
    for suf in allowed_suffixes:
        folder = base_folder / f"{scenario}{suf}"  # <-- join here

        if folder.exists():
            for tif in folder.glob("*.tif"):
                tif_files.append(tif)
                print(f"Found: {tif}")
        else:
            print(f"Missing folder: {folder}")

print("\nTotal .tif files found:", len(tif_files))

Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45y\p50_2030.tif
Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45y\p50_2018.tif
Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45y\p50_2040.tif
Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45y\p50_2050.tif
Missing folder: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sb2y
Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sm2y\p50_2030.tif
Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sm2y\p50_2040.tif
Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sm2y\p50_2050.tif
Missing folder: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sb2rb1y
Found: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sm2rb

In [9]:
# Files starting with P50
p_files = [p for p in tif_files if p.name.lower().startswith(probability)]

# Baseline subset (those containing "18")
baseline = [p for p in p_files if "18" in p.as_posix()]

# Remove baseline from p_files
p_files = [p for p in p_files if p not in baseline]

from pathlib import Path
import shutil

# DESTINATION ROOTS
sta_root = Path(r"N:/Deltabox/Postbox/Athanasiou, Panos/Salinity_Mekong/projections_gridded/salinity")
baseline_root = Path(r"N:/Deltabox/Postbox/Athanasiou, Panos/Salinity_Mekong/projections_gridded/salinity/baseline")

# COPY NORMAL FILES (p_files)
for src in p_files:
    folder_name = src.parent.name      # cc45y, cc85y, etc.
    dst_folder = sta_root / folder_name
    dst_folder.mkdir(parents=True, exist_ok=True)

    dst = dst_folder / src.name
    print(f"Copying NORMAL\n  {src}\n→ {dst}\n")
    shutil.copy2(src, dst)

# COPY BASELINE FILES (NO SUBFOLDER)
for src in baseline:
    baseline_root.mkdir(parents=True, exist_ok=True)

    dst = baseline_root / src.name
    print(f"Copying BASELINE\n  {src}\n→ {dst}\n")
    shutil.copy2(src, dst)

print("DONE.")

Copying NORMAL
  N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45y\p50_2030.tif
→ N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc45y\p50_2030.tif

Copying NORMAL
  N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45y\p50_2040.tif
→ N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc45y\p50_2040.tif

Copying NORMAL
  N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45y\p50_2050.tif
→ N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc45y\p50_2050.tif

Copying NORMAL
  N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sm2y\p50_2030.tif
→ N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc45sm2y\p50_2030.tif

Copying NORMAL
  N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\cc45sm2y\p50_2040.tif
→ N:\Deltabox\Postbox

In [10]:
from pathlib import Path
import rasterio
import numpy as np
import shutil

# Paths
stac = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity")
stac_out = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity_increase")

baseline_folder = stac / "baseline"

# Find baseline GeoTIFF (assume one file)
baseline_tif = list(baseline_folder.glob("*.tif"))[0]

print("Using baseline:", baseline_tif)

# Read baseline raster
with rasterio.open(baseline_tif) as src:
    baseline_data = src.read(1)
    baseline_profile = src.profile.copy()

# Process all other folders (everything except baseline)
for folder in stac.iterdir():
    if folder.name == "baseline":
        continue  # skip baseline folder

    if not folder.is_dir():
        continue

    # Output folder structure
    out_folder = stac_out / folder.name
    out_folder.mkdir(parents=True, exist_ok=True)

    # For each tif inside this subfolder
    for tif in folder.glob("*.tif"):
        print(f"Processing: {tif}")

        # Read input raster
        with rasterio.open(tif) as src:
            data = src.read(1)
            profile = src.profile

        # Subtract baseline
        out_data = data - baseline_data

        # Save output
        out_tif = out_folder / tif.name
        with rasterio.open(out_tif, 'w', **profile) as dst:
            dst.write(out_data, 1)

        print(f"Saved: {out_tif}")

print("DONE.")


Using baseline: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\baseline\p50_2018.tif
Processing: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc85sb2rb3y\p50_2030.tif
Saved: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity_increase\cc85sb2rb3y\p50_2030.tif
Processing: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc85sb2rb3y\p50_2040.tif
Saved: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity_increase\cc85sb2rb3y\p50_2040.tif
Processing: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc85sb2rb3y\p50_2050.tif
Saved: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity_increase\cc85sb2rb3y\p50_2050.tif
Processing: N:\Deltabox\Postbox\Athanasiou, Panos\Salinity_Mekong\projections_gridded\salinity\cc45sm2rb1y\p50_2040.tif
Saved: N:\Deltabox\Postbox\